# Parameters

In [ ]:
from common.utils.parameters import (
    mcwf_dt_from_scales,
    omega_c,
    scaled_N_Gamma,
)
from common.utils.phases import default_three_phase_protocol
from post_analysis import compute_mfe_residuals
from parser.moments import MomentSeries
from parser.moments import MomentParameters

from pathlib import Path

%reload_ext autoreload
%autoreload 2

output_dir = Path("output")

### Parameters

In [ ]:
# common fixed
Gamma = 1; dt = 1e-2; num_snapshots=100
# param
dN = 0
Ni = [10,10]
omega_i = [0.7]
ntraj = 100
# Model and parameters
Omega_factor = 0.1
Omega0 = scaled_N_Gamma(Omega_factor, sum(Ni), Gamma)
delta_factor = 0.02
delta0 = scaled_N_Gamma(delta_factor, sum(Ni), Gamma)
dt = mcwf_dt_from_scales(Omega0, delta0, sum(Ni), Gamma)
# protocol phases
phases = default_three_phase_protocol(
    T1=10.0,
    T2=10.0,
    T3=10.0,
    delta0=delta0,
    Omega0=Omega0,
)

total_time = float(sum(phase.duration for phase in phases))

Omega_c = omega_c(sum(Ni) // 2, Gamma)
print(f"Omega0 = {Omega0}")
print(f"Omega/Omega_c ratio = {Omega0 / Omega_c}")
print(f"delta0 = {delta0}")
print(f"dt = {dt}")


### Mean-field Equations

In [ ]:
from solvers.mfe import compute_mfe_j_moments, solve_mfe
from parser.mfe import MFESolverParameters

# moment initialization
mfe_moments = MomentSeries(
    num_snapshots=num_snapshots,
    total_time=total_time,
    )

# Define method parameters and solve using mean-field equations
mfe_parameters = MFESolverParameters(
    Ni=tuple(Ni),
    omega_i=tuple(omega_i),
    Gamma=Gamma,
    phases=phases,
)
mfe_result = solve_mfe(
    mfe_parameters,
    t_eval=mfe_moments.t,
)

# compute moments
mfe_moments.J = compute_mfe_j_moments(mfe_result)

### Monte-Carlo Wave Function

In [ ]:
from parser.mcwf import MCWFSolverParameters
from solvers.mcwf.ensamble_sim import run_trajectory_ensemble
from solvers.mcwf.j_moments import compute_mcwf_j_moments

# moment initialization
mcwf_moments = MomentSeries(
    num_snapshots=num_snapshots,
    total_time=total_time,
    )

# Define method parameters and simulate using Monte-Carlo wave function method
mcwf_parameters = MCWFSolverParameters(
    Ni=Ni,
    dN=dN,
    omega_i=omega_i,
    Gamma=Gamma,
    phases=phases,
    sector_distribution="binomial",
    dt=dt,
    shifted_jump_operator=True,
)
mcwf_ensemble = run_trajectory_ensemble(
    mcwf_parameters,
    t_eval=mcwf_moments.t,
    seed=1234,
    ntraj=ntraj,
    n_processes=-1,
    )

# compute moments
mcwf_moments.J = compute_mcwf_j_moments(
    mcwf_ensemble,
    n_processes=-1,
)

mcwf_moments.mfe_residuals = compute_mfe_residuals(
    mcwf_moments.J,
    parameters=MomentParameters(
        Gamma=Gamma,
        phases=phases,
        omega_groups=(0.7,1.3),
        N_groups=Ni
    )
)

### Qutip (mesolve & mcsolve)

In [ ]:
from parser.qutip import QutipMCSolverParameters, QutipMESolverParameters
from solvers.qutip_fixed_nj.sim import (
    simulate_fixed_nj_mc_trajectory,
    simulate_fixed_nj_me_trajectory
)
from solvers.qutip_fixed_nj.j_moments import compute_qutip_j_moments

# moment initialization
me_qutip_moments = MomentSeries(
    num_snapshots=num_snapshots,
    total_time=total_time,
    )
mc_qutip_moments = MomentSeries(
    num_snapshots=num_snapshots,
    total_time=total_time,
    )

# define method parameters and solve using Qutip 
#   - master equation solver
#   - quantum trajectory solver
me_qutip_parameters = QutipMESolverParameters(
    Ni=tuple(Ni),
    omega_i=tuple(omega_i),
    Gamma=Gamma,
    phases=phases,
    shifted_jump_operator=True,
)
mc_qutip_parameters = QutipMCSolverParameters(
    Ni=tuple(Ni),
    omega_i=tuple(omega_i),
    Gamma=Gamma,
    phases=phases,
    shifted_jump_operator=True,
)
me_qutip_ensamble = simulate_fixed_nj_me_trajectory(
    me_qutip_parameters,
    num_points=600,
)
mc_qutip_ensamble = simulate_fixed_nj_mc_trajectory(
    mc_qutip_parameters,
    num_points=600,
    ntraj=ntraj,
    seed=1234,
    n_processes=-1,
)

# compute moments
me_qutip_moments.J = compute_qutip_j_moments(
    me_qutip_ensamble,
)
mc_qutip_moments.J = compute_qutip_j_moments(
    mc_qutip_ensamble,
)

### Plotting

In [ ]:
from common.plotting import (
    plot_bloch_angles,
    plot_spin_components,
    plot_mfe_residuals
)

# Plot spin angles
fig, axes = plot_bloch_angles(
    mfe_moments.J,
    phases=phases,
    label="MFE",
    colour_family_index=None,
    shade_index=1,
    linestyle="-.",
)
fig, axes = plot_bloch_angles(
    mcwf_moments.J,
    phases=phases,
    label="MCWF",
    colour_family_index=None,
    shade_index=1,
    linestyle="--",
    axes=axes
)
fig, axes = plot_bloch_angles(
    mc_qutip_moments.J,
    phases=phases,
    label="MCQutip",
    colour_family_index=None,
    shade_index=1,
    linestyle=":",
    axes=axes
)
fig, axes = plot_bloch_angles(
    mc_qutip_moments.J,
    phases=phases,
    label="MEQutip",
    colour_family_index=None,
    shade_index=1,
    linestyle="-",
    axes=axes
)

# Plot spin components
fig, axes = plot_spin_components(
    mfe_moments.J,
    normalized=False,
    phases=phases,
    label="MFE",
    colour_family_index=None,
    shade_index=1,
    linestyle="-.",
)
fig, axes = plot_spin_components(
    mcwf_moments.J,
    normalized=False,
    phases=phases,
    label="MCWF",
    colour_family_index=None,
    shade_index=1,
    linestyle="--",
    axes=axes,
)
fig, axes = plot_spin_components(
    mc_qutip_moments.J,
    normalized=False,
    phases=phases,
    label="MCQutip",
    colour_family_index=None,
    shade_index=1,
    linestyle=":",
    axes=axes
)
fig, axes = plot_spin_components(
    mc_qutip_moments.J,
    normalized=False,
    phases=phases,
    label="MEQutip",
    colour_family_index=None,
    shade_index=1,
    linestyle="-",
    axes=axes
)


# plot mfe residuals
fig, axes = plot_mfe_residuals(
    mcwf_moments.mfe_residuals,
    phases=phases,
    label="MCWF",
    colour_index=0,
    linestyle="-",
)